# Data Cleaning



The first notebook section will essentially answer:

How do we reproducibly take the raw Excel workbook containing two yearly sheets and create one transaction-level working dataset without altering the data yet?

In [2]:
#importing libraries

import pandas as pd

In [5]:
#Define the raw data path

RAW_FILE = "../data/raw/online_retail_II.xlsx"

In [6]:
#Load both worksheets

df_2009_2010 = pd.read_excel(
    RAW_FILE,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    RAW_FILE,
    sheet_name="Year 2010-2011"
)

In [7]:
#Check that ingestion worked


print("2009-2010 shape:", df_2009_2010.shape)
print("2010-2011 shape:", df_2010_2011.shape)

2009-2010 shape: (525461, 8)
2010-2011 shape: (541910, 8)


In [8]:
#Check the columns

print("2009-2010 columns:")
print(df_2009_2010.columns.tolist())

print("\n2010-2011 columns:")
print(df_2010_2011.columns.tolist())

2009-2010 columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

2010-2011 columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [9]:
#Combine the two datasets

df_raw = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

In [10]:
#Check the shape

print("Combined shape:", df_raw.shape)

Combined shape: (1067371, 8)


In [11]:
#Confirm the combined structure  

df_raw.head()

df_raw.info()

df_raw.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


(1067371, 8)

## 4.1 Raw Data Ingestion & Workbook Combination

The source dataset is stored as an Excel workbook containing two worksheets:

* `Year 2009-2010`
* `Year 2010-2011`

Both worksheets contain the same eight transaction-level fields and represent data at the transaction-line grain.

The two worksheets are loaded independently and then combined vertically into a single working dataset using a reproducible ingestion process.

At this stage, no cleaning, filtering, classification, deduplication, or value transformation is applied. The purpose of this step is to establish a combined representation of the raw source data while preserving the original transaction records.

The resulting combined dataset contains:

* **1,067,371 transaction-line records**
* **8 source fields**

The combined dataset is retained in its raw structural form and will serve as the starting point for the subsequent cleaning and transformation steps.

### Ingestion Principle

> **Ingest first, transform second.**

Keeping ingestion separate from cleaning allows changes introduced by later processing stages to be distinguished from the original source data.


In [12]:
df_clean = df_raw.rename(
    columns={
        "Invoice": "invoice",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "invoice_date",
        "Price": "unit_price",
        "Customer ID": "customer_id",
        "Country": "country"
    }
)

In [13]:
#Verify the new schema

df_clean.columns.tolist()

['invoice',
 'stock_code',
 'description',
 'quantity',
 'invoice_date',
 'unit_price',
 'customer_id',
 'country']

In [14]:
df_clean.head()

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [15]:
print("Shape:", df_clean.shape)

Shape: (1067371, 8)


## 4.2 Schema & Column Standardization

The combined raw dataset uses the original column names supplied by the source workbook. These names are standardized to a consistent lowercase `snake_case` convention before further transformation.

The target schema is:

| Source Column | Target Column  | Purpose                                 |
| ------------- | -------------- | --------------------------------------- |
| `Invoice`     | `invoice`      | Transaction/invoice identifier          |
| `StockCode`   | `stock_code`   | Product or transaction code             |
| `Description` | `description`  | Product or transaction description      |
| `Quantity`    | `quantity`     | Transaction quantity                    |
| `InvoiceDate` | `invoice_date` | Transaction date and time               |
| `Price`       | `unit_price`   | Recorded unit price                     |
| `Customer ID` | `customer_id`  | Customer identifier                     |
| `Country`     | `country`      | Country associated with the transaction |

The `Price` field is renamed to `unit_price` to make its business meaning explicit and distinguish it from other possible price concepts.

No values are modified during this step. The transformation is limited to column-name standardization.

The dataset should retain the same:

* **1,067,371 transaction-line records**
* **8 columns**

This establishes a consistent schema for the subsequent data-cleaning and transformation stages.


In [16]:
#Inspect the current types

df_clean.dtypes

invoice                 object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id            float64
country                 object
dtype: object

In [17]:
#Standardize identifier/text fields

df_clean["invoice"] = df_clean["invoice"].astype("string")
df_clean["stock_code"] = df_clean["stock_code"].astype("string")
df_clean["description"] = df_clean["description"].astype("string")
df_clean["country"] = df_clean["country"].astype("string")

In [18]:
#Standardize customer ID

df_clean["customer_id"] = df_clean["customer_id"].astype("Int64")

In [19]:
#Standardize quantity

df_clean["quantity"] = df_clean["quantity"].astype("Int64")

In [20]:
#Standardize unit price

df_clean["unit_price"] = pd.to_numeric(
    df_clean["unit_price"],
    errors="coerce"
)

In [21]:
#Confirm invoice date

df_clean["invoice_date"] = pd.to_datetime(
    df_clean["invoice_date"],
    errors="coerce"
)

In [22]:
#Inspect the resulting schema

df_clean.dtypes

invoice         string[python]
stock_code      string[python]
description     string[python]
quantity                 Int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id              Int64
country         string[python]
dtype: object

In [23]:
print(df_clean.shape)

(1067371, 8)


## 4.3 Data Type Standardization

The standardized column names are now assigned explicit data types according to their business roles.

Identifier and categorical fields such as `invoice`, `stock_code`, `description`, and `country` are represented using Pandas string types.

The `customer_id` field is represented using Pandas' nullable `Int64` type. This preserves customer identifiers as integers while allowing missing customer IDs to remain missing. Customer IDs are treated as identifiers rather than numerical measurements.

The `quantity` field is represented using nullable integer values because transaction quantities are whole units and the target model must be capable of representing missing values without converting the field into floating-point values.

The `unit_price` field is converted to a numeric representation. Values that cannot be interpreted numerically are converted to missing values for subsequent data-quality validation rather than being replaced with an assumed value.

The `invoice_date` field is explicitly converted to a datetime representation. Values that cannot be parsed as valid dates are converted to missing values for subsequent validation.

No records are intentionally removed during this step.

The resulting dataset should retain:

* **1,067,371 transaction-line records**
* **8 standardized fields**

This step establishes the technical data types required for the subsequent cleaning, validation, classification, and reporting stages.


In [24]:
#Establish the current row count

initial_row_count = len(df_clean)

print("Initial row count:", initial_row_count)

Initial row count: 1067371


In [25]:
#Identify exact duplicate rows

duplicate_mask = df_clean.duplicated(keep="first")

print("Exact duplicate rows identified:", duplicate_mask.sum())

Exact duplicate rows identified: 34335


In [26]:
#Inspect the duplicates before removing them

duplicate_rows = df_clean[duplicate_mask]

duplicate_rows.head(20)

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
390,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
657,489529,22028,PENNY FARTHING BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984,United Kingdom
658,489529,22036,DINOSAUR BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984,United Kingdom


In [27]:
duplicate_rows.shape

(34335, 8)

In [28]:
#How many duplicate groups exist

print(
    "Unique duplicated row patterns:",
    df_clean[df_clean.duplicated(keep=False)].drop_duplicates().shape[0]
)

Unique duplicated row patterns: 32907


In [29]:
#Check whether invoice repetition is involved

print(
    "Rows with repeated invoice values:",
    df_clean["invoice"].duplicated(keep=False).sum()
)

Rows with repeated invoice values: 1054077


In [30]:
#Remove only exact duplicate rows

df_clean = df_clean.drop_duplicates(keep="first").reset_index(drop=True)

In [31]:
#Record result

cleaned_row_count = len(df_clean)
duplicates_removed = initial_row_count - cleaned_row_count

print("Initial row count:", initial_row_count)
print("Duplicates removed:", duplicates_removed)
print("Cleaned row count:", cleaned_row_count)

Initial row count: 1067371
Duplicates removed: 34335
Cleaned row count: 1033036


## 4.4 Duplicate Handling

Duplicate handling is performed at the transaction-line level.

The source dataset contains repeated invoice identifiers because a single invoice can contain multiple transaction lines. Therefore, `invoice` is not used as a unique row identifier and repeated invoice values are not treated as duplicates.

The cleaning process identifies exact duplicate transaction rows using all standardized fields.

For each set of identical rows, the first occurrence is retained and subsequent identical occurrences are removed.

The duplicate-removal process records an audit trail containing:

* initial row count
* number of exact duplicate rows identified
* number of duplicate rows removed
* resulting cleaned row count

No duplicate removal is performed based solely on invoice number, customer ID, stock code, or any other individual field.

This approach preserves legitimate multi-line invoices while removing redundant copies of identical transaction records.

The duplicate rule is intentionally conservative: only transaction lines that are completely identical after schema and type standardization are removed. Potential business duplicates that differ in one or more fields are not automatically deleted and may require separate investigation.

### Duplicate Handling Principle

> **Repeated identifiers are not necessarily duplicate transactions.**

The objective is to remove redundant transaction records without changing legitimate purchasing activity.


In [32]:
print("Duplicate rows remaining:",
      df_clean.duplicated(keep=False).sum())

Duplicate rows remaining: 0
